# Paper 6 — Step 0: Local data preparation

**Broken Windows, Broken Trust: Collective Efficacy, Repeat Complaining, and Municipal
Responsiveness in 30 Million 311 Service Requests**

This notebook runs **on your own machine** (the raw files are 17 GB in total and cannot be
uploaded). It reduces every raw source in `Paper6/DataPaper6/` to a small set of analysis-ready
tables in `Paper6/derived/` (a few hundred MB at most), which the analysis notebook then uses.

| Stage | Input | Output |
|---|---|---|
| 0 | environment check | — |
| 1 | TIGER tract boundaries (downloaded) | `tract_geo.parquet` |
| 2 | NHGIS ACS 2020–2024 5-year | `acs_tract.parquet` |
| 3 | NYC 311 (14.7 GB, streamed) | `nyc311_events.parquet` (compact), `nyc311_typedesc.parquet` |
| 4 | NYPD complaints 2020–2024 | `nyc_crime_events.parquet` |
| 5 | Chicago 311 (12 legacy files) | `chi311_events.parquet` |
| 6 | Zillow ZHVI + Redfin | `zhvi_zip_month.parquet`, `redfin_city_month.parquet` |
| 7 | assembly | `nyc_tract_month.parquet`, `chi_tract_month.parquet`, `manifest.json` |

**Do not run the heavy stages inside this Cursor notebook if you plan to close Cursor.**
A notebook kernel dies with the IDE. Use the detached runner instead (it keeps going after
you close Cursor, and writes live progress you can read when you reopen):

```bash
# once, from BehaveP6/
python scripts/run_job.py start prep     # CPU: CSV stream + GeoPandas
python scripts/run_job.py status         # reopen Cursor anytime and run this
```

Or run the **next code cell** (“start / watch”). Stage 3 now checkpoints every chunk, so a
killed job resumes instead of starting the 14 GB file over.

**Device**: prep is CPU/IO (GPU does not accelerate pandas/GeoPandas). Analysis LightGBM and
the optional GNN use a GPU only when one has enough free VRAM; this lab node’s GPUs are often
busy, so those jobs fall back to CPU automatically.

**Runtime**: roughly 45–90 minutes, dominated by Stage 3. Finished stage files are skipped.

> Nothing here modifies your raw data. All writes go to `derived/`.

## Stage 0 — Configuration and environment

In [ ]:
import os, sys, json, time, hashlib, warnings, zipfile, io, gc
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT = Path(os.environ.get("PAPER6_ROOT", Path.cwd())).resolve()

RAW      = PROJECT / "DataPaper6"
DERIVED  = PROJECT / "derived"
TMP      = PROJECT / "derived" / "_tmp"
GEO      = PROJECT / "derived" / "_geo"
for p in (DERIVED, TMP, GEO):
    p.mkdir(parents=True, exist_ok=True)

# Study window (NYPD extract and the ACS 5-year vintage both cover 2020-2024)
YEAR_MIN, YEAR_MAX = 2020, 2024
# Chicago legacy 311 files cover roughly 2011-2019; keep their own window
CHI_YEAR_MIN, CHI_YEAR_MAX = 2011, 2019

CHUNK = 400_000          # rows per chunk when streaming the 14.7 GB file
try:
    _avail = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_AVPHYS_PAGES")
    if _avail > 64 * (1 << 30):
        CHUNK = 800_000  # lab node with plenty of RAM
except (ValueError, OSError, AttributeError):
    pass
COORD_DP = 4             # coordinate rounding for the point-in-polygon cache (~11 m)

print("python :", sys.version.split()[0])
print("project:", PROJECT, "exists =", PROJECT.exists())
print("raw    :", RAW, "exists =", RAW.exists())
print("chunk  :", f"{CHUNK:,}")

In [ ]:
# Dependency check. Install anything missing, then re-run this cell.
need = {}
for mod, pipname in [("pandas","pandas"), ("numpy","numpy"), ("pyarrow","pyarrow"),
                     ("geopandas","geopandas"), ("shapely","shapely"), ("requests","requests"),
                     ("tqdm","tqdm")]:
    try:
        __import__(mod)
    except ImportError:
        need[mod] = pipname
if need:
    print("MISSING:", need)
    print("Run:  pip install " + " ".join(need.values()))
else:
    import pandas as pd, numpy as np, geopandas as gpd
    print("pandas", pd.__version__, "| numpy", np.__version__, "| geopandas", gpd.__version__)

In [ ]:
import pandas as pd, numpy as np
import geopandas as gpd
from shapely.geometry import Point
import requests
from tqdm.auto import tqdm

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

def norm(s):
    "Normalise a column name: lowercase, alphanumeric + underscore only."
    return "".join(ch if ch.isalnum() else "_" for ch in str(s).strip().lower()).strip("_")

def pick(cols_norm, *cands):
    "Return the first candidate present in a {normalised: original} mapping."
    for c in cands:
        if c in cols_norm:
            return cols_norm[c]
    return None

def hash64(series):
    "Stable 64-bit hash of a string Series (address keys)."
    return pd.util.hash_pandas_object(series.fillna("").astype(str), index=False).astype("int64")

# NYC 311 exports dates as '01/23/2020 04:05:06 PM'. Trying the known format first is an
# order of magnitude faster than letting pandas infer row by row.
_FMTS = ["%m/%d/%Y %I:%M:%S %p", "%m/%d/%Y %H:%M:%S", "%m/%d/%Y", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"]
def to_dt(s):
    s = s.astype(str)
    for fmt in _FMTS:
        out = pd.to_datetime(s, errors="coerce", format=fmt)
        if out.notna().mean() > 0.5:
            return out
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(s, errors="coerce")

def month_floor(dt_series):
    return dt_series.dt.to_period("M").dt.to_timestamp()

def done(path):
    p = Path(path)
    if p.exists():
        print(f"[skip] {p.name} already exists ({p.stat().st_size/1e6:.1f} MB)")
        return True
    return False

def stamp(msg, t0):
    print(f"[{time.strftime('%H:%M:%S')}] {msg} — {time.time()-t0:.1f}s")

## Start / watch (survives closing Cursor)

Run the cell below **instead of** stepping through Stages 1–7 in this kernel if the job might
outlive this Cursor session. It starts `scripts/run_prep.py` in a detached process (systemd
user unit, or `setsid` fallback). Re-run the same cell after you reopen Cursor to see the
stage, percent, and last log lines. Prep stays on **CPU**; GPU is reserved for LightGBM / GNN.

In [ ]:
# Start detached prep (if not already running) and print live status.
# Safe to re-run after reopening Cursor — it will not launch a second copy.
import subprocess
from pathlib import Path as _P

_py = PROJECT / ".venv" / "bin" / "python"
if not _py.exists():
    _py = _P(sys.executable)
_job = PROJECT / "scripts" / "run_job.py"

def _run(*args):
    r = subprocess.run([str(_py), str(_job), *args], cwd=str(PROJECT), capture_output=True, text=True)
    print(r.stdout or r.stderr)
    return r

st = _run("status")
sys.path.insert(0, str(PROJECT / "scripts"))
from paper6_runtime import pid_alive, read_status
_cur = read_status(PROJECT)
_alive = bool(_cur) and pid_alive(_cur.get("pid"))
_done = bool(_cur) and _cur.get("state") == "done"
_has_panel = (DERIVED / "chi_tract_month.parquet").exists() or (DERIVED / "nyc_tract_month.parquet").exists()
if not _alive and not _has_panel:
    print("\n--- starting detached prep ---")
    _run("start", "prep")
    print("\n--- status ---")
    _run("status")
elif _has_panel:
    print("Prep outputs already present (Chicago and/or NYC panel). Not starting a new copy.")
else:
    print("(job running — not starting a new copy)")

## Stage 1 — Census tract geometry (TIGER/Line 2023)

The 2020–2024 ACS uses 2020 census-tract boundaries. We download the tract shapefiles for
New York (FIPS 36) and Illinois (FIPS 17) once and cache them under `derived/_geo/`.
If your network blocks `census.gov`, download the two `.zip` files manually and drop them in
that folder — the cell will find them.

In [ ]:
TIGER = {"36": "tl_2023_36_tract.zip", "17": "tl_2023_17_tract.zip"}
BASE  = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/"
UA = {"User-Agent": "Paper6-research/1.0 (academic; TIGER tract download)"}

def download_tiger_zip(fn, dst):
    """HTTPS first; www2.census.gov often 403s lab IPs — FTP still serves the same files."""
    try:
        r = requests.get(BASE + fn, timeout=120, stream=True, headers=UA)
        if r.status_code == 200:
            with open(dst, "wb") as f:
                for blk in r.iter_content(1 << 20):
                    f.write(blk)
            return "https"
        print(f"  HTTPS {r.status_code}; trying FTP...")
    except Exception as exc:
        print(f"  HTTPS failed ({exc}); trying FTP...")
    from ftplib import FTP
    ftp = FTP("ftp2.census.gov", timeout=60)
    ftp.login()
    ftp.cwd("/geo/tiger/TIGER2023/TRACT")
    with open(dst, "wb") as f:
        ftp.retrbinary(f"RETR {fn}", f.write, blocksize=1 << 20)
    ftp.quit()
    return "ftp"

for fips, fn in TIGER.items():
    dst = GEO / fn
    if dst.exists():
        print("[cached]", fn); continue
    print("downloading", fn, "...")
    via = download_tiger_zip(fn, dst)
    print("  ->", round(dst.stat().st_size/1e6, 1), "MB via", via)

In [ ]:
out = DERIVED / "tract_geo.parquet"
if not done(out):
    parts = []
    for fips, fn in TIGER.items():
        g = gpd.read_file(f"zip://{(GEO/fn).as_posix()}")
        g = g[["GEOID", "STATEFP", "COUNTYFP", "TRACTCE", "ALAND", "AWATER", "INTPTLAT", "INTPTLON", "geometry"]]
        parts.append(g)
    tracts = pd.concat(parts, ignore_index=True)
    tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4269").to_crs("EPSG:4326")
    tracts.to_file(GEO / "tracts_ny_il.gpkg", driver="GPKG")     # cached geometry for the sjoins
    flat = pd.DataFrame({
        "GEOID":   tracts["GEOID"].astype(str),
        "STATEFP": tracts["STATEFP"].astype(str),
        "COUNTYFP":tracts["COUNTYFP"].astype(str),
        "ALAND":   tracts["ALAND"].astype("float64"),
        "lat":     tracts["INTPTLAT"].astype(float),
        "lon":     tracts["INTPTLON"].astype(float),
    })
    NYC_COUNTIES = {"005","047","061","081","085"}   # Bronx, Kings, New York, Queens, Richmond
    CHI_COUNTY   = {"031"}                            # Cook
    flat["city"] = np.where(flat.STATEFP.eq("36") & flat.COUNTYFP.isin(NYC_COUNTIES), "NYC",
                    np.where(flat.STATEFP.eq("17") & flat.COUNTYFP.isin(CHI_COUNTY), "CHI", "other"))
    flat["area_km2"] = flat["ALAND"] / 1e6
    flat.to_parquet(out, index=False)
    print(flat.city.value_counts())
tract_geo = pd.read_parquet(out)
print(tract_geo.shape); tract_geo.head(3)

In [ ]:
# Point-in-polygon helper, shared by every stage that has lat/lon.
_TRACTS_GDF = None
def tracts_gdf():
    global _TRACTS_GDF
    if _TRACTS_GDF is None:
        _TRACTS_GDF = gpd.read_file(GEO / "tracts_ny_il.gpkg")[["GEOID", "geometry"]]
        _TRACTS_GDF = _TRACTS_GDF.to_crs("EPSG:4326")
    return _TRACTS_GDF

def coords_to_tract(unique_coords: pd.DataFrame) -> pd.DataFrame:
    # unique_coords: columns lat_k, lon_k (rounded ints = coord x 10**COORD_DP).
    # Returns lat_k, lon_k, GEOID.
    f = 10 ** COORD_DP
    pts = gpd.GeoDataFrame(
        unique_coords.copy(),
        geometry=gpd.points_from_xy(unique_coords.lon_k / f, unique_coords.lat_k / f),
        crs="EPSG:4326")
    j = gpd.sjoin(pts, tracts_gdf(), how="left", predicate="within")
    return j[["lat_k", "lon_k", "GEOID"]].drop_duplicates(subset=["lat_k", "lon_k"])

## Stage 2 — ACS 2020–2024 tract covariates (NHGIS)

We keep the classic neighbourhood-structure predictors used in the collective-efficacy
literature: population size and density, educational attainment, unemployment, residential
stability (owner-occupancy) and vacancy (the direct Broken-Windows structural analogue).

In [ ]:
out = DERIVED / "acs_tract.parquet"
if not done(out):
    acs_dir = next(p for p in RAW.iterdir() if p.is_dir() and p.name.startswith("2020_2024_ACS5"))
    src = next(acs_dir.glob("*_tract.csv"))
    keep_ctx = ["GISJOIN","STATEA","COUNTYA","TRACTA","STATE","COUNTY","TL_GEO_ID"]
    hdr = pd.read_csv(src, nrows=0)
    est = [c for c in hdr.columns if c.startswith(("AUO6E","AUQ8E","AUTWE","AUUCE","AUUDE","AUUEE"))]
    a = pd.read_csv(src, usecols=keep_ctx + est, dtype={"STATEA":str,"COUNTYA":str,"TRACTA":str})
    a["GEOID"] = a.STATEA.str.zfill(2) + a.COUNTYA.str.zfill(3) + a.TRACTA.str.zfill(6)

    pop      = a["AUO6E001"].astype(float)
    edu_tot  = a["AUQ8E001"].astype(float)
    ba_plus  = a[[f"AUQ8E{n:03d}" for n in (22,23,24,25)]].astype(float).sum(axis=1)
    lt_hs    = a[[f"AUQ8E{n:03d}" for n in range(2,17)]].astype(float).sum(axis=1)
    lf       = a["AUTWE003"].astype(float)                   # civilian labour force
    unemp    = a["AUTWE005"].astype(float)
    hu       = a["AUUCE001"].astype(float)
    occ      = a["AUUDE002"].astype(float)
    vac      = a["AUUDE003"].astype(float)
    own      = a["AUUEE002"].astype(float)
    rent     = a["AUUEE003"].astype(float)

    acs = pd.DataFrame({
        "GEOID": a["GEOID"],
        "state": a["STATE"], "county": a["COUNTY"],
        "pop": pop,
        "pct_ba_plus":     np.where(edu_tot > 0, 100 * ba_plus / edu_tot, np.nan),
        "pct_lt_hs":       np.where(edu_tot > 0, 100 * lt_hs  / edu_tot, np.nan),
        "pct_unemployed":  np.where(lf > 0,      100 * unemp  / lf,      np.nan),
        "housing_units":   hu,
        "pct_vacant":      np.where(hu > 0,      100 * vac / hu,         np.nan),
        "pct_owner_occ":   np.where((own+rent) > 0, 100 * own / (own+rent), np.nan),
        "pct_renter_occ":  np.where((own+rent) > 0, 100 * rent/ (own+rent), np.nan),
    })
    acs = acs.merge(tract_geo[["GEOID","area_km2","city","lat","lon"]], on="GEOID", how="left")
    acs["pop_density_km2"] = acs["pop"] / acs["area_km2"].replace(0, np.nan)
    acs.to_parquet(out, index=False)
acs = pd.read_parquet(out)
print(acs.groupby("city").size())
acs[acs.city=="NYC"].describe().T[["mean","std","min","max"]].round(2)

## Stage 3 — NYC 311 (the 14.7 GB file)

Streamed in chunks. For every request we keep only what the analysis needs and write a
**compact event table** (one row per request, ~10 narrow columns) plus a frequency table of the
`complaint_type | descriptor` vocabulary, which is the input to the text/NLP layer.

Two theory-driven codings are applied here:

* **Complaint domain** — physical disorder, social disorder, housing habitability, infrastructure,
  sanitation, other. Physical + social disorder are the Broken-Windows signal.
* **Reporting channel** — phone vs. self-service (online / mobile). Channel choice is our
  observable analogue of *voice* effort in the Exit–Voice–Loyalty frame.

In [ ]:
# Theory-driven mapping from complaint_type keywords to domains.
DOMAIN_RULES = [
    ("physical_disorder", ["graffiti","dirty condition","illegal dumping","derelict vehicle",
                           "abandoned vehicle","street condition","sidewalk condition","curb condition",
                           "damaged tree","dead tree","overgrown tree","vacant lot","abandoned building",
                           "unsanitary pigeon","litter basket","dumpster","broken parking meter",
                           "rodent","standing water"]),
    ("social_disorder",   ["noise","drug activity","illegal fireworks","panhandling","homeless",
                           "encampment","disorderly youth","drinking","urinating in public",
                           "graffiti - vandalism","illegal parking","blocked driveway","animal abuse",
                           "unlicensed vendor","consumer complaint"]),
    ("housing",           ["heat/hot water","heating","plumbing","paint","plaster","door/window",
                           "electric","flooring","appliance","general construction","water leak",
                           "unsanitary condition","mold","elevator","safety","outside building"]),
    ("infrastructure",    ["street light","traffic signal","water system","sewer","street sign",
                           "highway condition","bridge","root/sewer","catch basin","hydrant",
                           "traffic","broken muni meter","curb"]),
    ("sanitation",        ["missed collection","sanitation condition","recycling","electronics waste",
                           "commercial disposal","residential disposal","dumpster complaint",
                           "sweeping","snow","request large bulky item collection"]),
]
DOMAINS = ["physical_disorder","social_disorder","housing","infrastructure","sanitation","other"]
DOM_ID  = {d:i for i,d in enumerate(DOMAINS)}

_DOM_CACHE = {}
def domain_of(ctype: str) -> str:
    t = (ctype or "")
    if t in _DOM_CACHE:
        return _DOM_CACHE[t]
    tl = t.lower()
    res = "other"
    for dom, keys in DOMAIN_RULES:
        if any(k in tl for k in keys):
            res = dom; break
    _DOM_CACHE[t] = res
    return res

def domain_ids(ct: pd.Series) -> np.ndarray:
    "Vectorised: resolve the rule set once per distinct complaint type, then map."
    u = pd.Index(ct.unique())
    lut = pd.Series([DOM_ID[domain_of(x)] for x in u], index=u, dtype="int8")
    return ct.map(lut).fillna(DOM_ID["other"]).astype("int8").values

CHANNEL_ID = {"phone":0, "online":1, "mobile":2, "unknown":3, "other":4}
def channel_of(x):
    t = str(x or "").strip().lower()
    if t.startswith("phone"):  return 0
    if t.startswith("online"): return 1
    if t.startswith("mobile"): return 2
    if t.startswith("unknown"):return 3
    return 4

def channel_ids(s: pd.Series) -> np.ndarray:
    u = pd.Index(s.astype(str).unique())
    lut = pd.Series([channel_of(x) for x in u], index=u, dtype="int8")
    return s.astype(str).map(lut).fillna(4).astype("int8").values

In [ ]:
NYC_SRC = RAW / "311-service-requests-from-2010-to-present-001.csv"
SKIP_NYC311 = (not NYC_SRC.exists()) or os.environ.get("PAPER6_SKIP_NYC311", "").strip().lower() in {"1", "true", "yes"}
if SKIP_NYC311:
    print("[skip] NYC 311 CSV missing — Chicago-only. Later NYC cells are no-ops.")
    print("Expected:", NYC_SRC)
else:
    print(NYC_SRC, f"{NYC_SRC.stat().st_size/1e9:.1f} GB")
if not SKIP_NYC311:

    def _probe_nyc_years(path, n_windows=7):
        size = path.stat().st_size
        years = set()
        with path.open("rb") as f:
            for i in range(n_windows):
                pos = int(size * i / max(n_windows - 1, 1))
                f.seek(pos); f.readline()
                for _ in range(80):
                    line = f.readline()
                    if not line: break
                    parts = line.split(b",", 2)
                    if len(parts) < 2: continue
                    y = parts[1][:4]
                    if y.isdigit(): years.add(int(y))
        return years

    _nyc_years = _probe_nyc_years(NYC_SRC)
    print("NYC 311 years sampled:", sorted(_nyc_years))
    assert any(YEAR_MIN <= y <= YEAR_MAX for y in _nyc_years), (
        f"NYC 311 has no dates in {YEAR_MIN}-{YEAR_MAX} (sampled {sorted(_nyc_years)}). "
        "The 14.7 GB Windows copy is a duplicated 2019 extract — re-download 2020-2024 "
        "before this stage (see DataPaper6/README.md).")

    hdr = pd.read_csv(NYC_SRC, nrows=3)
    cn  = {norm(c): c for c in hdr.columns}
    print(len(hdr.columns), "columns")
    for c in hdr.columns: print("  -", c)

In [ ]:
if SKIP_NYC311:
    COL = {}
    print("[skip] column map — no NYC 311 file")
else:
    COL = dict(
        uk      = pick(cn, "unique_key"),
        created = pick(cn, "created_date"),
        closed  = pick(cn, "closed_date"),
        agency  = pick(cn, "agency"),
        ctype   = pick(cn, "complaint_type"),
        descr   = pick(cn, "descriptor"),
        zipc    = pick(cn, "incident_zip"),
        addr    = pick(cn, "incident_address", "street_name"),
        status  = pick(cn, "status"),
        boro    = pick(cn, "borough"),
        channel = pick(cn, "open_data_channel_type"),
        lat     = pick(cn, "latitude"),
        lon     = pick(cn, "longitude"),
        bbl     = pick(cn, "bbl"),
        cb      = pick(cn, "community_board"),
    )
    missing = [k for k, v in COL.items() if v is None]
    print("resolved:", {k: v for k, v in COL.items() if v})
    assert not [k for k in ("created","ctype","lat","lon") if COL[k] is None], f"critical columns missing: {missing}"
    print("optional missing:", missing)

In [ ]:
if SKIP_NYC311:
    print("[skip] NYC 311 stream — file not on disk")
else:
    out_ev  = DERIVED / "nyc311_events.parquet"
    out_td  = DERIVED / "nyc311_typedesc.parquet"

    if not (out_ev.exists() and out_td.exists()):
        t0 = time.time()
        usecols = [v for v in COL.values() if v]
        part_dir = TMP / "nyc311_parts"; part_dir.mkdir(exist_ok=True)
        import pickle
        progress_path = TMP / "nyc311_progress.json"
        td_path = TMP / "nyc311_td.pkl"
        progress = {"next_chunk": 0, "n_rows": 0, "n_kept": 0}
        td_counter = {}
        if progress_path.exists() and td_path.exists() and any(part_dir.glob("p*.parquet")):
            progress = json.loads(progress_path.read_text(encoding="utf-8"))
            with td_path.open("rb") as _f:
                td_counter = pickle.load(_f)
            print(f"[resume] NYC 311 from chunk {progress['next_chunk']} "
                  f"(read {progress['n_rows']:,}, kept {progress['n_kept']:,})")
        else:
            for old in part_dir.glob("*.parquet"):
                old.unlink()

        n_rows = int(progress.get("n_rows", 0))
        n_kept = int(progress.get("n_kept", 0))
        start_i = int(progress.get("next_chunk", 0))
        f = 10 ** COORD_DP
        _skip = n_rows
        reader = pd.read_csv(NYC_SRC, usecols=usecols, chunksize=CHUNK,
                             dtype=str, low_memory=False, on_bad_lines="skip",
                             skiprows=(lambda i, s=_skip: 0 < i <= s) if _skip else None)

        # ---------- pass 1: stream, code, and write compact chunk files -------------
        for j, ch in enumerate(reader):
            i = start_i + j
            n_rows += len(ch)
            created = to_dt(ch[COL["created"]])
            m = created.dt.year.between(YEAR_MIN, YEAR_MAX)

            # vocabulary frequencies use the whole file, not only the study window
            ct_all = ch[COL["ctype"]].fillna("").str.strip()
            de_all = (ch[COL["descr"]].fillna("").str.strip()
                      if COL["descr"] else pd.Series([""] * len(ch), index=ch.index))
            vc = pd.DataFrame({"a": ct_all, "b": de_all}).value_counts()
            for k, v in vc.items():
                td_counter[k] = td_counter.get(k, 0) + int(v)

            ch = ch[m]; created = created[m]
            if len(ch) == 0:
                continue
            closed = to_dt(ch[COL["closed"]]) if COL["closed"] else pd.Series(pd.NaT, index=ch.index)

            lat = pd.to_numeric(ch[COL["lat"]], errors="coerce")
            lon = pd.to_numeric(ch[COL["lon"]], errors="coerce")
            ok  = lat.between(40.4, 41.1) & lon.between(-74.3, -73.6)
            lat_k = np.where(ok, np.round(lat * f), np.nan)
            lon_k = np.where(ok, np.round(lon * f), np.nan)

            ct = ch[COL["ctype"]].fillna("").str.strip()
            blank = pd.Series([""] * len(ch), index=ch.index)
            addr_src = (ch[COL["bbl"]].fillna("") if COL["bbl"] else blank)
            addr_src = addr_src.where(addr_src.str.len() > 5,
                                      (ch[COL["addr"]].fillna("").str.upper().str.strip()
                                       if COL["addr"] else blank))
            addr_src = addr_src.where(addr_src.str.len() > 2,
                                      pd.Series(lat_k, index=ch.index).astype(str) + "_" +
                                      pd.Series(lon_k, index=ch.index).astype(str))

            ev = pd.DataFrame({
                "created": created.values,
                "closed":  closed.values,
                "lat_k":   lat_k, "lon_k": lon_k,
                "ctype":   ct.values,
                "domain":  domain_ids(ct),
                "agency":  (ch[COL["agency"]].fillna("").values if COL["agency"] else ""),
                "zipc":    (pd.to_numeric(ch[COL["zipc"]], errors="coerce").values if COL["zipc"] else np.nan),
                "channel": (channel_ids(ch[COL["channel"]]) if COL["channel"] else np.int8(3)),
                "status":  (ch[COL["status"]].fillna("").values if COL["status"] else ""),
                "boro":    (ch[COL["boro"]].fillna("").values if COL["boro"] else ""),
                "addr_hash": hash64(addr_src).values,
            })
            for c in ("ctype", "agency", "boro", "status"):
                ev[c] = ev[c].astype("category")
            ev.to_parquet(part_dir / f"p{i:05d}.parquet", index=False)
            n_kept += len(ev)
            progress_path.write_text(json.dumps({"next_chunk": i + 1, "n_rows": n_rows, "n_kept": n_kept}))
            with td_path.open("wb") as _f:
                pickle.dump(td_counter, _f)
            if i % 10 == 0:
                print(f"  chunk {i:>4} | read {n_rows/1e6:6.2f}M | kept {n_kept/1e6:6.2f}M | "
                      f"{time.time()-t0:6.0f}s", flush=True)
            del ev, ch, created, closed; gc.collect()

        stamp(f"pass 1 complete: {n_rows:,} rows read, {n_kept:,} in window", t0)

        td = (pd.Series(td_counter).rename("n").rename_axis(["complaint_type", "descriptor"])
                .reset_index().sort_values("n", ascending=False))
        td["domain"] = [domain_of(x) for x in td.complaint_type]
        td.to_parquet(out_td, index=False)
        print("vocabulary:", td.shape, "| unique complaint_type:", td.complaint_type.nunique())

        # ---------- unique coordinates -> census tract ------------------------------
        parts = sorted(part_dir.glob("*.parquet"))
        coords = pd.concat([pd.read_parquet(p, columns=["lat_k", "lon_k"]) for p in parts],
                           ignore_index=True).dropna().drop_duplicates()
        coords = coords.astype("int64").reset_index(drop=True)
        print("unique coordinates:", f"{len(coords):,}")
        lut = coords_to_tract(coords)
        lut.to_parquet(TMP / "nyc_coord_tract.parquet", index=False)
        stamp("tract lookup built", t0)

        # ---------- pass 2: join the tract id and stream into one parquet -----------
        import pyarrow as pa, pyarrow.parquet as pq
        writer = None; matched = 0; total = 0
        for p in parts:
            d = pd.read_parquet(p)
            d["lat_k"] = d["lat_k"].astype("Int64"); d["lon_k"] = d["lon_k"].astype("Int64")
            d = d.merge(lut.astype({"lat_k": "Int64", "lon_k": "Int64"}), on=["lat_k", "lon_k"], how="left")
            for c in ("ctype", "agency", "boro", "status"):
                d[c] = d[c].astype(str)
            matched += d.GEOID.notna().sum(); total += len(d)
            tbl = pa.Table.from_pandas(d, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(out_ev, tbl.schema, compression="zstd")
            writer.write_table(tbl)
            del d, tbl; gc.collect()
        writer.close()
        stamp(f"events written: {total:,} rows, tract matched {matched/max(total,1):.1%}", t0)
        for p in parts: p.unlink()

    print("events :", out_ev.stat().st_size/1e6, "MB")
    print("vocab  :", out_td.stat().st_size/1e6, "MB")

In [ ]:
if SKIP_NYC311 or not (DERIVED / "nyc311_typedesc.parquet").exists():
    print("[skip] nyc311_typedesc.parquet — NYC 311 not downloaded")
    td = None
else:
    td = pd.read_parquet(DERIVED / "nyc311_typedesc.parquet")
    print(f"{td.n.sum():,} requests across {len(td):,} complaint_type x descriptor pairs")
    display(td.head(15))
    display(td.groupby("domain").n.sum().sort_values(ascending=False).to_frame("requests"))

## Stage 4 — NYPD complaints 2020–2024 → tract-month crime panel

Only offence classification, date and location are used. Suspect and victim demographic
columns are deliberately dropped and never enter the analysis.

In [ ]:
out = DERIVED / "nyc_crime_events.parquet"
if not done(out):
    t0 = time.time(); frames = []
    files = sorted((RAW / "NYPD_2020_2024").glob("*.csv"))
    f = 10 ** COORD_DP
    coord_seen = set()
    for src in files:
        h = pd.read_csv(src, nrows=3); cnn = {norm(c): c for c in h.columns}
        c_dt  = pick(cnn, "cmplnt_fr_dt", "rpt_dt", "complaint_date")
        c_cat = pick(cnn, "law_cat_cd", "law_category")
        c_ofn = pick(cnn, "ofns_desc", "offense_description")
        c_lat = pick(cnn, "latitude"); c_lon = pick(cnn, "longitude")
        cols  = [c for c in (c_dt, c_cat, c_ofn, c_lat, c_lon) if c]
        d = pd.read_csv(src, usecols=cols, dtype=str, low_memory=False, on_bad_lines="skip")
        dt = to_dt(d[c_dt])
        lat = pd.to_numeric(d[c_lat], errors="coerce"); lon = pd.to_numeric(d[c_lon], errors="coerce")
        ok = lat.between(40.4, 41.1) & lon.between(-74.3, -73.6) & dt.dt.year.between(YEAR_MIN, YEAR_MAX)
        d = pd.DataFrame({
            "created": dt[ok].values,
            "lat_k": np.round(lat[ok] * f).values, "lon_k": np.round(lon[ok] * f).values,
            "law_cat": d[c_cat][ok].fillna("").str.upper().values if c_cat else "",
            "ofns":    d[c_ofn][ok].fillna("").str.upper().values if c_ofn else "",
        })
        frames.append(d)
        coord_seen.update(map(tuple, d[["lat_k","lon_k"]].dropna().astype("int64").values))
        print(f"  {src.name}: {len(d):,} usable rows", flush=True)
    cr = pd.concat(frames, ignore_index=True)
    lut = coords_to_tract(pd.DataFrame(sorted(coord_seen), columns=["lat_k","lon_k"]))
    cr = cr.merge(lut, on=["lat_k","lon_k"], how="left")

    VIOLENT = ("ASSAULT","ROBBERY","MURDER","HOMICIDE","RAPE","FELONY ASSAULT","STRANGULATION",
               "KIDNAPPING","ARSON","WEAPONS")
    PROPERTY= ("BURGLARY","GRAND LARCENY","PETIT LARCENY","CRIMINAL MISCHIEF","THEFT",
               "POSSESSION OF STOLEN","FORGERY","FRAUD")
    cr["violent"]  = cr.ofns.str.contains("|".join(VIOLENT),  na=False).astype("int8")
    cr["property"] = cr.ofns.str.contains("|".join(PROPERTY), na=False).astype("int8")
    cr["ofns"] = cr["ofns"].astype("category"); cr["law_cat"] = cr["law_cat"].astype("category")
    cr.to_parquet(out, index=False)
    stamp(f"crime events: {len(cr):,}, tract matched {cr.GEOID.notna().mean():.1%}", t0)
crime = pd.read_parquet(out)
print(crime.shape); print(crime.law_cat.value_counts())

## Stage 5 — Chicago 311 (cross-city replication)

The twelve legacy Chicago files have inconsistent headers, so column names are resolved
case-insensitively per file. Chicago supplies the replication sample for RQ1–RQ2 only
(no matching crime extract was downloaded).

In [ ]:
out = DERIVED / "chi311_events.parquet"
CHI_TYPE_DOMAIN = {
    "graffiti removal": "physical_disorder", "abandoned vehicle": "physical_disorder",
    "vacant/abandoned building": "physical_disorder", "tree debris": "physical_disorder",
    "rodent baiting": "physical_disorder", "sanitation code violation": "sanitation",
    "garbage cart": "sanitation", "pot hole": "infrastructure",
    "street light - all/out": "infrastructure", "street light - 1/out": "infrastructure",
    "alley light out": "infrastructure", "tree trim": "infrastructure",
}
def chi_domain(fname):
    s = fname.lower()
    if "graffiti" in s or "abandoned-vehicles" in s or "vacant" in s or "tree-debris" in s or "rodent" in s:
        return "physical_disorder"
    if "sanitation" in s or "garbage" in s:
        return "sanitation"
    return "infrastructure"

if not done(out):
    t0 = time.time(); frames = []; coord_seen = set(); f = 10 ** COORD_DP
    for src in sorted((RAW / "Chicago_311_Service_Requests").glob("*.csv")):
        h = pd.read_csv(src, nrows=3); cnn = {norm(c): c for c in h.columns}
        c_cr = pick(cnn, "creation_date", "date_service_request_was_received")
        c_cm = pick(cnn, "completion_date")
        c_ty = pick(cnn, "type_of_service_request", "service_request_type")
        c_st = pick(cnn, "status")
        c_zp = pick(cnn, "zip_code", "zip_code_1")
        c_ca = pick(cnn, "community_area")
        c_wd = pick(cnn, "ward")
        c_ad = pick(cnn, "street_address", "address_street_name")
        c_la = pick(cnn, "latitude"); c_lo = pick(cnn, "longitude")
        cols = [c for c in (c_cr,c_cm,c_ty,c_st,c_zp,c_ca,c_wd,c_ad,c_la,c_lo) if c]
        d = pd.read_csv(src, usecols=cols, dtype=str, low_memory=False, on_bad_lines="skip")
        cr = to_dt(d[c_cr])
        cm = to_dt(d[c_cm]) if c_cm else pd.Series(pd.NaT, index=d.index)
        la = pd.to_numeric(d[c_la], errors="coerce") if c_la else pd.Series(np.nan, index=d.index)
        lo = pd.to_numeric(d[c_lo], errors="coerce") if c_lo else pd.Series(np.nan, index=d.index)
        ok = cr.dt.year.between(CHI_YEAR_MIN, CHI_YEAR_MAX)
        okxy = la.between(41.6, 42.1) & lo.between(-87.95, -87.5)
        addr = (d[c_ad].fillna("").str.upper().str.strip() if c_ad else pd.Series("", index=d.index))
        e = pd.DataFrame({
            "created": cr[ok].values,
            "closed":  cm[ok].values,
            "lat_k": np.where(okxy[ok], np.round(la[ok]*f), np.nan),
            "lon_k": np.where(okxy[ok], np.round(lo[ok]*f), np.nan),
            "srtype": (d[c_ty][ok].fillna("").values if c_ty else src.stem),
            "domain": chi_domain(src.name),
            "status": (d[c_st][ok].fillna("").values if c_st else ""),
            "zipc":   (pd.to_numeric(d[c_zp][ok], errors="coerce").values if c_zp else np.nan),
            "comm_area": (pd.to_numeric(d[c_ca][ok], errors="coerce").values if c_ca else np.nan),
            "ward":   (pd.to_numeric(d[c_wd][ok], errors="coerce").values if c_wd else np.nan),
            "addr_hash": hash64(addr[ok]).values,
            "src": src.stem,
        })
        frames.append(e)
        coord_seen.update(map(tuple, e[["lat_k","lon_k"]].dropna().astype("int64").values))
        print(f"  {src.name:<62} {len(e):>9,} rows", flush=True)
    chi = pd.concat(frames, ignore_index=True)
    lut = coords_to_tract(pd.DataFrame(sorted(coord_seen), columns=["lat_k","lon_k"]))
    chi = chi.merge(lut, on=["lat_k","lon_k"], how="left")
    for c in ("srtype","domain","status","src"): chi[c] = chi[c].astype("category")
    chi.to_parquet(out, index=False)
    stamp(f"chicago events: {len(chi):,}, tract matched {chi.GEOID.notna().mean():.1%}", t0)
chi = pd.read_parquet(out); print(chi.shape); print(chi.domain.value_counts())

## Stage 6 — Housing market (Zillow ZHVI, Redfin)

In [ ]:
out = DERIVED / "zhvi_zip_month.parquet"
if not done(out):
    src = next((RAW / "Zillow&Redfin").glob("Zip_zhvi*.csv"))
    z = pd.read_csv(src, dtype={"RegionName": str}, low_memory=False)
    id_cols = [c for c in z.columns if not c[:4].isdigit()]
    val_cols = [c for c in z.columns if c[:4].isdigit()]
    keep = z[z.State.isin(["NY","IL"])]
    zl = keep.melt(id_vars=[c for c in ("RegionName","State","City","CountyName") if c in keep.columns],
                   value_vars=val_cols, var_name="date", value_name="zhvi")
    zl["date"] = pd.to_datetime(zl["date"], errors="coerce")
    zl = zl.dropna(subset=["zhvi", "date"])
    zl["month"] = month_floor(zl["date"])
    zl["zipc"] = pd.to_numeric(zl["RegionName"], errors="coerce")
    zl = zl[["zipc","State","City","CountyName","month","zhvi"]]
    zl.to_parquet(out, index=False)
zhvi = pd.read_parquet(out); print(zhvi.shape, zhvi.month.min(), zhvi.month.max())

out = DERIVED / "redfin_city_month.parquet"
if not done(out):
    src = next((RAW / "Zillow&Redfin").glob("redfin_*.csv"))
    r = pd.read_csv(src)
    r.columns = [norm(c) for c in r.columns]
    r["month"] = month_floor(pd.to_datetime(r["period_begin"], errors="coerce"))
    r.to_parquet(out, index=False)
print(pd.read_parquet(out).shape)

## Stage 7 — Assemble the tract-month panels

This is where the theoretical constructs become variables:

* **Disorder signal** — physical / social disorder complaint counts and rates.
* **Municipal responsiveness** — median days-to-close, share resolved within 3 days,
  share unresolved after 30 days, all measured at the tract-month level.
* **Repeat complaining** — the behavioural outcome: complaints from addresses that already
  complained in the previous 12 months, recurrence of the same domain at the same address,
  and the concentration of complaints across addresses (Herfindahl index).
* **Voice channel mix** — share of self-service (online/mobile) reports.

In [ ]:
def build_panel(ev, unit="GEOID", window=(YEAR_MIN, YEAR_MAX), label="nyc"):
    ev = ev.dropna(subset=[unit]).copy()
    ev["month"] = month_floor(ev["created"])
    ev["days_to_close"] = (ev["closed"] - ev["created"]).dt.total_seconds() / 86400
    ev.loc[(ev.days_to_close < 0) | (ev.days_to_close > 365), "days_to_close"] = np.nan

    # ---- repeat behaviour: previous complaint at the same address within 365 days
    ev = ev.sort_values(["addr_hash", "created"])
    prev = ev.groupby("addr_hash", observed=True)["created"].shift(1)
    gap = (ev["created"] - prev).dt.total_seconds() / 86400
    ev["is_repeat_12m"] = ((gap <= 365) & gap.notna()).astype("int8")
    prev_d = ev.groupby(["addr_hash", "domain"], observed=True)["created"].shift(1)
    gapd = (ev["created"] - prev_d).dt.total_seconds() / 86400
    ev["is_same_domain_repeat"] = ((gapd <= 365) & gapd.notna()).astype("int8")

    # vectorised responsiveness indicators (much faster than groupby-lambdas)
    d = ev["days_to_close"]
    ev["_closed3"]  = (d <= 3).astype("float32").where(d.notna())
    ev["_open30"]   = (d > 30).astype("float32").where(d.notna())
    ev["_unres"]    = d.isna().astype("float32")

    g = ev.groupby([unit, "month"], observed=True)
    panel = g.agg(
        n_complaints        = ("created", "size"),
        n_addresses         = ("addr_hash", "nunique"),
        median_days_close   = ("days_to_close", "median"),
        mean_days_close     = ("days_to_close", "mean"),
        share_closed_3d     = ("_closed3", "mean"),
        share_open_30d      = ("_open30", "mean"),
        share_unresolved    = ("_unres", "mean"),
        n_repeat_12m        = ("is_repeat_12m", "sum"),
        n_same_domain_rep   = ("is_same_domain_repeat", "sum"),
    ).reset_index()
    p90 = g["days_to_close"].quantile(0.90).rename("p90_days_close").reset_index()
    panel = panel.merge(p90, on=[unit, "month"], how="left")

    # domain composition
    dom = ev["domain"]
    if pd.api.types.is_numeric_dtype(dom):
        dom = dom.map({i: d for d, i in DOM_ID.items()})
    ev["_dom"] = dom.astype(str)
    wide = (ev.groupby([unit, "month", "_dom"], observed=True).size()
              .unstack("_dom", fill_value=0).add_prefix("n_").reset_index())
    panel = panel.merge(wide, on=[unit, "month"], how="left")

    # channel mix (NYC only)
    if "channel" in ev.columns:
        ch = (ev.assign(selfserve=ev["channel"].isin([1, 2]).astype("int8"))
                .groupby([unit, "month"], observed=True)["selfserve"].mean()
                .rename("share_selfserve").reset_index())
        panel = panel.merge(ch, on=[unit, "month"], how="left")

    # address concentration (Herfindahl over addresses within tract-month)
    cnt = ev.groupby([unit, "month", "addr_hash"], observed=True).size().rename("k").reset_index()
    hhi = (cnt.assign(sh=lambda d: d.k / d.groupby([unit, "month"])["k"].transform("sum"))
              .assign(sh2=lambda d: d.sh ** 2)
              .groupby([unit, "month"])["sh2"].sum().rename("addr_hhi").reset_index())
    panel = panel.merge(hhi, on=[unit, "month"], how="left")

    panel["repeat_rate"]      = panel.n_repeat_12m / panel.n_complaints
    panel["same_domain_rate"] = panel.n_same_domain_rep / panel.n_complaints
    panel["complaints_per_address"] = panel.n_complaints / panel.n_addresses
    for c in [c for c in panel.columns if c.startswith("n_") and c not in
              ("n_complaints","n_addresses","n_repeat_12m","n_same_domain_rep")]:
        panel[c.replace("n_", "share_", 1)] = panel[c] / panel.n_complaints
    panel["city_label"] = label
    return panel

In [ ]:
if SKIP_NYC311 or not (DERIVED / "nyc311_events.parquet").exists():
    print("[skip] nyc_tract_month.parquet — no NYC 311 events")
else:
    out = DERIVED / "nyc_tract_month.parquet"
    if not done(out):
        t0 = time.time()
        ev = pd.read_parquet(DERIVED / "nyc311_events.parquet")
        panel = build_panel(ev, unit="GEOID", label="nyc")

        # crime outcomes at tract-month
        cr = pd.read_parquet(DERIVED / "nyc_crime_events.parquet").dropna(subset=["GEOID"])
        cr["month"] = month_floor(cr["created"])
        cg = cr.groupby(["GEOID","month"]).agg(
                n_crime=("created","size"),
                n_violent=("violent","sum"),
                n_property=("property","sum"),
                n_felony=("law_cat", lambda s: (s == "FELONY").sum())).reset_index()
        panel = panel.merge(cg, on=["GEOID","month"], how="left")

        # dominant ZIP per tract, from the 311 data itself -> ZHVI link
        zmap = (ev.dropna(subset=["GEOID","zipc"]).groupby(["GEOID","zipc"]).size()
                  .rename("k").reset_index().sort_values("k", ascending=False)
                  .drop_duplicates("GEOID")[["GEOID","zipc"]])
        panel = panel.merge(zmap, on="GEOID", how="left")
        zh = pd.read_parquet(DERIVED / "zhvi_zip_month.parquet")[["zipc","month","zhvi"]]
        panel = panel.merge(zh, on=["zipc","month"], how="left")

        panel = panel.merge(pd.read_parquet(DERIVED / "acs_tract.parquet"), on="GEOID", how="left")
        panel = panel[panel.city.eq("NYC")]
        panel.to_parquet(out, index=False)
        stamp(f"NYC panel: {panel.shape}", t0)
        del ev; gc.collect()
    nyc_panel = pd.read_parquet(out)
    print(nyc_panel.shape, "| tracts:", nyc_panel.GEOID.nunique(), "| months:", nyc_panel.month.nunique())
    nyc_panel.head(3)

In [ ]:
out = DERIVED / "chi_tract_month.parquet"
if not done(out):
    chi = pd.read_parquet(DERIVED / "chi311_events.parquet")
    p = build_panel(chi, unit="GEOID", window=(CHI_YEAR_MIN, CHI_YEAR_MAX), label="chi")
    p = p.merge(pd.read_parquet(DERIVED / "acs_tract.parquet"), on="GEOID", how="left")
    p = p[p.city.eq("CHI")]
    p.to_parquet(out, index=False)
    print("Chicago panel:", p.shape)
chi_panel = pd.read_parquet(out)
print(chi_panel.shape, "| tracts:", chi_panel.GEOID.nunique())

In [ ]:
if SKIP_NYC311 or not (DERIVED / "nyc311_events.parquet").exists():
    print("[skip] nyc_agency_month.parquet — no NYC 311 events")
else:
    # Agency-month responsiveness series (input to the difference-in-differences design)
    out = DERIVED / "nyc_agency_month.parquet"
    if not done(out):
        ev = pd.read_parquet(DERIVED / "nyc311_events.parquet",
                             columns=["created","closed","agency","GEOID","domain"])
        ev["month"] = month_floor(ev["created"])
        ev["days"] = (ev["closed"] - ev["created"]).dt.total_seconds()/86400
        ev.loc[(ev.days < 0) | (ev.days > 365), "days"] = np.nan
        ev["_c3"] = (ev.days <= 3).astype("float32").where(ev.days.notna())
        ev["_un"] = ev.days.isna().astype("float32")
        ag = ev.groupby(["agency","month"], observed=True).agg(
            n=("created","size"), median_days=("days","median"),
            share_3d=("_c3","mean"), share_unresolved=("_un","mean")).reset_index()
        ag.to_parquet(out, index=False)
        del ev; gc.collect()
    print(pd.read_parquet(out).shape)

In [ ]:
manifest = {}
for p in sorted(DERIVED.glob("*.parquet")):
    d = pd.read_parquet(p)
    manifest[p.name] = {"rows": int(len(d)), "cols": list(map(str, d.columns)),
                        "mb": round(p.stat().st_size/1e6, 2)}
    del d
manifest["_meta"] = {"built": time.strftime("%Y-%m-%d %H:%M:%S"),
                     "window_nyc": [YEAR_MIN, YEAR_MAX], "window_chi": [CHI_YEAR_MIN, CHI_YEAR_MAX]}
(DERIVED / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
for k, v in manifest.items():
    if k != "_meta":
        print(f"{k:<34} {v['rows']:>10,} rows  {v['mb']:>8.1f} MB")
print("\nDONE. Everything the analysis needs is in:", DERIVED)

### When this finishes

### When this finishes

Tell Claude *"prep xong rồi"* — the files in `derived/` are small enough to be read
directly from this session, and the analysis notebook (`Paper6_Analysis.ipynb`) takes over
from there.

If you used the detached runner, start analysis the same way (it will pick a GPU if one is
free, otherwise CPU):

```bash
python scripts/run_job.py start analysis
python scripts/run_job.py status
```

If any stage fails, copy the error message across: each stage is independent and resumable
(Stage 3 also resumes mid-file), so only the failing stage has to be re-run.